# 06 · Service vs weather (GTFS-enhanced)

Este notebook responde **Q3** do plano: se a chuva está associada a piora operacional em
`headway_p50`, `speed_p50`, `service_gap_index` e `observed_vs_scheduled_trip_ratio`.

**Janela usada aqui:** a janela integrada de **19 dias** com ticket + clima + mobilidade.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 120})


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / 'data' / 'derived').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root from the current working directory.')


PROJECT_ROOT = locate_project_root()
DERIVED = PROJECT_ROOT / 'data' / 'derived'
FIGURES = DERIVED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
WEATHER_ORDER = ['Clear', 'Light Rain', 'Moderate Rain', 'Heavy Rain / Storm']


def savefig(name: str) -> Path:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path


In [ ]:
integrated = pd.read_parquet(DERIVED / 'integrated_route_hour.parquet')
quality = pd.read_csv(DERIVED / 'mobility_day_quality_flags.csv', parse_dates=['date'])

service_all = integrated.dropna(subset=['observed_trip_count']).copy()
service_all['weather_bucket'] = pd.Categorical(
    np.where(
        service_all['rain_mm'].fillna(0) >= 10,
        'Adverse (>=10 mm)',
        np.where(service_all['rain_mm'].fillna(0) > 0, 'Rain (<10 mm)', 'Clear'),
    ),
    categories=['Clear', 'Rain (<10 mm)', 'Adverse (>=10 mm)'],
    ordered=True,
)
service_clean = service_all.loc[service_all['coverage_flag'].fillna('ok') != 'partial_day'].copy()

print('=== Integrated service sample ===')
print('unique integrated days:', pd.to_datetime(service_all['date']).dt.date.nunique())
print('service rows (all):', len(service_all))
print('service rows (excluding partial-day flags):', len(service_clean))
print('flagged mobility days:', quality.loc[quality['is_partial_day'], 'date'].dt.strftime('%Y-%m-%d').tolist())
print()

summary = service_clean.groupby('weather_bucket', observed=False)[['headway_p50', 'speed_p50', 'service_gap_index', 'observed_vs_scheduled_trip_ratio']].agg(['count', 'mean', 'median'])
print('=== Weather-bucket summary on clean sample ===')
print(summary.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.boxplot(data=service_clean, x='weather_bucket', y='headway_p50', ax=axes[0])
axes[0].set_title('Headway p50 by weather bucket')
sns.boxplot(data=service_clean, x='weather_bucket', y='speed_p50', ax=axes[1])
axes[1].set_title('Speed p50 by weather bucket')
sns.boxplot(data=service_clean, x='weather_bucket', y='service_gap_index', ax=axes[2])
axes[2].set_title('Service gap index by weather bucket')
for ax in axes:
    ax.tick_params(axis='x', rotation=15)
savefig('q3_service_boxplots.png')
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=service_clean, x='weather_bucket', y='observed_vs_scheduled_trip_ratio', ax=ax)
ax.set_title('Observed / scheduled trip ratio by weather bucket')
ax.tick_params(axis='x', rotation=15)
savefig('q3_schedule_ratio_by_weather.png')
plt.show()


In [ ]:
def fit_service_models(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    output = []
    specs = {
        'headway_p50': frame.dropna(subset=['headway_p50']).copy(),
        'speed_p50': frame.dropna(subset=['speed_p50']).copy(),
        'service_gap_index': frame.dropna(subset=['service_gap_index']).copy(),
    }
    for metric, df_metric in specs.items():
        model = smf.ols(
            f'{metric} ~ rain_mm + C(route_norm) + C(hour) + C(day_of_week)',
            data=df_metric,
        ).fit(cov_type='HC1')
        output.append(
            {
                'sample': label,
                'metric': metric,
                'rain_coef': model.params['rain_mm'],
                'rain_pvalue': model.pvalues['rain_mm'],
                'nobs': int(model.nobs),
            }
        )
    return pd.DataFrame(output)


sensitivity = pd.concat(
    [
        fit_service_models(service_all, 'all mobility days'),
        fit_service_models(service_clean, 'excluding partial days'),
    ],
    ignore_index=True,
)
print('=== Sensitivity comparison ===')
print(sensitivity.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=sensitivity, x='metric', y='rain_coef', hue='sample', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Q3 · Rain coefficient with vs without partial-day mobility flags')
ax.set_ylabel('Coefficient on rain_mm')
ax.set_xlabel('')
savefig('q3_sensitivity_flagged_days.png')
plt.show()


## Leitura preliminar

- O notebook reporta a amostra operacional na **janela integrada de 19 dias**.
- A sensibilidade **com vs sem** dias parciais foi explicitamente rodada, em vez de apenas filtrar silenciosamente.
- As conclusões de Q3 devem ser formuladas para **chuva leve/moderada**, porque não há base observacional para tempestades severas nessa janela.
